In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset
import re
from tqdm import tqdm
import nltk
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
nltk.download('stopwords', quiet=True)
english_stopwords = set(stopwords.words('english'))

print(" Libraries loaded successfully")

## 1. Load Dataset



In [ ]:

print("Loading dataset from HuggingFace...")
dataset = load_dataset('handeyilmaz/job-descriptions-final', split='train')
df = pd.DataFrame(dataset)


print(f"\nDataset shape: {df.shape}")
print(f"Unique positions: {df['position'].nunique():,}")
print(f"\nTop 10 positions:")
print(df['position'].value_counts().head(10))

## 2. Position Cleaning Function



In [ ]:
def clean_position(pos):
 """
 Clean position text for rule-based matching.

 Steps:
 1. Lowercase
 2. Remove numbers
 3. Remove special characters
 4. Keep whole words (NO stemming)
 5. Normalize common variations
 """
 if pd.isnull(pos):
 return ''

 # Lowercase
 pos = str(pos).lower()

 # Remove numbers
 pos = re.sub(r'\d+', '', pos)

 # Remove special characters except spaces and dots (for .net, etc.)
 pos = re.sub(r'[^\w\s.]', ' ', pos)

 # Normalize common variations BEFORE removing extra spaces
 pos = pos.replace('.net', 'dotnet')
 pos = pos.replace('c++', 'cpp')
 pos = pos.replace('c#', 'csharp')

 # Remove extra whitespace
 pos = ' '.join(pos.split())



 return pos

In [ ]:
# Apply cleaning to all positions
print("Cleaning all positions...")
tqdm.pandas(desc="Cleaning")
df['position_cleaned'] = df['position'].progress_apply(clean_position)

print(f"\n Cleaning complete!")
print(f"Sample cleaned positions:")
print(df[['position', 'position_cleaned']].head(10))

## 3. Define Category Mapping Rules



In [ ]:
# Define categories in PRIORITY ORDER (top categories checked first)
CATEGORY_KEYWORDS = [
 # 1. EXECUTIVE/LEADERSHIP
 ('Executive/Leadership', [
 'ceo', 'cto', 'cfo', 'coo', 'cmo', 'ciso', 'cpo', 'cbdo',
 'chief', 'president', 'presid', 'vp', 'vice', 'founder',
 'co-found', 'owner', 'partner', 'board'
 ]),

 # 2. MANAGEMENT
 ('Management', [
 'manager', 'manag', 'head of', 'lead', 'supervisor', 'supervis',
 'director', 'direct', 'team lead', 'scrum master', 'producer',
 'game producer', 'pmo',
 'junior pm', 'middle pm', 'senior pm',
 ]),

 # 3. MOBILE DEVELOPMENT
 ('Mobile Development', [
 'ios', 'android', 'flutter', 'react native', 'swift', 'kotlin',
 'mobile', 'mobl', 'xamarin'
 ]),

 # 4. DEVOPS
 ('DevOps', [
 'devops', 'dev ops', 'sre', 'site reliab', 'infrastruct',
 'cloud engineer', 'aws', 'azure', 'gcp', 'kubernetes', 'docker',
 'mlops', 'devsecops', 'techops', 'dataops', 'cloudops', 'secops'
 ]),

 # 5. DATA/ANALYTICS
 ('Data/Analytics', [
 'data scientist', 'data scienc', 'analyst', 'analyt',
 'business intelligence', 'bi ', 'ml engineer', 'machine learn',
 'ai ', 'artificial intellig', 'statistic', 'statist',
 'business analyst', 'ba ', 'senior ba', 'middle ba', 'junior ba',
 'data entry', 'mathematician', 'game mathematician',
 'data annotation', 'data annotator',
 ]),

 # 6. QA/TESTING
 ('QA/Testing', [
 'qa', 'quality assurance', 'qualiti', 'tester', 'test engineer',
 'automat test', 'manual test', 'quality specialist',
 'manual qc', 'qc ', 'junior qc', 'middle qc', 'senior qc',
 ]),

 # 7. ENGINEERING/IT
 ('Engineering/IT', [
 'engineer', 'engin', 'developer', 'develop', 'programmer', 'program',
 'software', 'softwar', 'backend', 'frontend', 'front end', 'full stack', 'fullstack',
 'java', 'python', 'javascript', '.net', 'dotnet', 'php', 'ruby', 'golang', 'node.js', 'angular',
 'architect', 'tech lead',
 'dba', 'database administrator', 'senior dba',
 'security specialist', 'information security', 'it security',
 'webmaster', 'html coder', 'gamedev', 'game dev',
 'react dev', 'react.js', 'vue dev', 'front-end',
 'sap consultant', 'it specialist', 'integration specialist', 'application specialist',
 'security expert',
 # NEW from clusters:
 'embedded', 'automation', 'scala', # Cluster 0, 5, 6
 'atqc', # Cluster 5
 'dynamics 365', 'd365', 'business central', # Cluster 1, 9
 'application security', 'cyber security consultant', # Cluster 4
 'blockchain', 'crypto', 'solidity', 'cryptography', # Cluster 8, 15
 'mongodb', 'mongo db', # Cluster 18
 'html/css', 'css coder', # Cluster 28
 'nodejs', 'node js', # Cluster 35
 'vue.js', 'vue js', # Cluster 40
 'etl', 'dwh', # Cluster 24
 'integrator', # Cluster 21
 'rust', # Cluster 36
 'splunk', # Cluster 41
 'django', 'typescript', # Cluster 35
 'odoo', # Cluster 16
 'shopify', # Cluster 33
 'hubspot', # Cluster 14
 ]),

 # 8. DESIGN/CREATIVE
 ('Design/Creative', [
 'ux', 'ui', 'designer', 'design', 'graphic', 'visual',
 'creative', 'creat', 'art director', 'video', 'multimedia',
 'illustrator', 'illustr', '2d artist', '3d artist', 'artist',
 'animator', '2d animator', '3d animator', 'vfx', 'technical artist',
 'level artist', 'game artist', 'character artist',
 'copywriter',
 # NEW from clusters:
 '3d generalist', 'generalist', # Cluster 2
 '3d modeler', 'modeler', # Cluster 3
 'proofreader', 'editor', 'scriptwriter', # Cluster 12, 22
 ]),

 # 9. SALES/MARKETING
 ('Sales/Marketing', [
 'sales', 'sale', 'marketing', 'market', 'brand', 'account manager',
 'business development', 'busi develop', 'growth', 'seo', 'sem',
 'advertising', 'advertis', 'digital market', 'pr ', 'public relat',
 'ppc specialist', 'ppc', 'smm specialist', 'smm', 'media buyer',
 'content strategist', 'aso specialist',
 'outreach specialist',
 # NEW from clusters:
 'facebook ads', 'google ads', # Cluster 25
 'conversion optimization', 'cro specialist', # Cluster 13
 ]),

 # 10. HR/RECRUITMENT
 ('HR/Recruitment', [
 'hr ', 'human resourc', 'recruiter', 'recruit', 'talent',
 'hiring', 'hire', 'people oper', 'payroll', 'compensation', 'compens',
 'hrd', 'sourcer', 'sourcing specialist', 'agile coach',
 'l&d specialist',
 'c&b specialist', 'c&b',
 # NEW from clusters:
 'global mobility', 'relocation specialist', 'immigration', # Cluster 27
 ]),

 # 11. OPERATIONS/ADMIN
 ('Operations/Admin', [
 'operations', 'oper', 'admin', 'administrator', 'administr',
 'assistant', 'assist', 'coordinator', 'coordin', 'office',
 'clerk', 'reception', 'recept', 'support', 'customer service',
 'customer care', 'customer succes', 'logistics', 'logist',
 'supply chain', 'bookkeeper', 'helpdesk', 'it helpdesk',
 'kyc specialist', 'pma specialist',
 'secretary', 'executive secretary',
 # NEW from clusters:
 'moderator', 'content moderator', # Cluster 11
 'client success', 'client experience', # Cluster 10
 ]),

 # 12. FINANCE/LEGAL
 ('Finance/Legal', [
 'finance', 'financ', 'accountant', 'account', 'financial',
 'audit', 'tax', 'treasury', 'treasur', 'bank',
 'legal', 'lawyer', 'compliance', 'compli', 'risk',
 'billing', 'invoicing', 'sap fico', 'sap fi',
 # NEW from clusters:
 'ifrs', # Cluster 29 (International Financial Reporting Standards)
 ]),

 # 13. ACADEMIC/RESEARCH
 ('Academic/Research', [
 'research', 'professor', 'profess', 'lecturer', 'lectur',
 'academic', 'academ', 'scientist', 'phd', 'fellow',
 'instructor', 'instruct', 'postdoc', 'technical writer',
 'content writer', 'content editor',
 # NEW from clusters:
 'documentation writer', 'technical documentation', # Cluster 39
 'english teacher', 'native speaker', 'teacher', # Cluster 23
 'bigdata trainer', 'trainer', # Cluster 7
 ]),

 # 14. HEALTHCARE
 ('Healthcare', [
 'doctor', 'nurse', 'medical doctor', 'physician',
 'hospital', 'clinic', 'pharmacist', 'dentist',
 'health', 'patient care', 'healthcare',
 'hipaa',
 ]),
]

print(f" Defined {len(CATEGORY_KEYWORDS)} categories")
print("\nCategories (in priority order):")
for i, (category, keywords) in enumerate(CATEGORY_KEYWORDS, 1):
 print(f"{i:2d}. {category:<25} ({len(keywords)} keywords)")

## 4. Rule-Based Mapping Function

In [ ]:
def rule_based_mapping(pos_cleaned):

 if pd.isnull(pos_cleaned) or pos_cleaned == '':
 return 'Other'

 pos_cleaned = str(pos_cleaned).lower()

 # Check each category in priority order
 for category, keywords in CATEGORY_KEYWORDS:
 for keyword in keywords:
 if keyword in pos_cleaned:
 return category

 return 'Other'


print(" Rule-based mapping function defined!")

# Test rule-based mapping
test_positions = [
 "ceo",
 "senior manag python",
 "ios develop",
 "data scienc",
 "softwar engin",
 "ux design",
 "sale manag",
 "random position xyz",
]

print("\nTest rule-based mapping:")
print(f"{'Cleaned Position':<30} {'Category'}")
print("="*60)
for pos in test_positions:
 category = rule_based_mapping(pos)
 print(f"{pos:<30} {category}")

In [ ]:
# Apply rule-based mapping to all positions
print("Applying rule-based mapping to all positions...\n")
tqdm.pandas(desc="Mapping")
df['position_group'] = df['position_cleaned'].progress_apply(rule_based_mapping)

# Statistics
print(f"\n{'='*60}")
print("RULE-BASED MAPPING RESULTS")
print(f"{'='*60}")
print(f"\nCategory distribution:")
category_counts = df['position_group'].value_counts()
print(category_counts)

other_count = category_counts.get('Other', 0)
mapped_count = len(df) - other_count
print(f"\nMapped by rules: {mapped_count:,} ({mapped_count/len(df)*100:.2f}%)")
print(f"Remaining 'Other': {other_count:,} ({other_count/len(df)*100:.2f}%)")
print(f"{'='*60}")

## 5. Manual Verification, Sample Review



In [ ]:
# Show sample positions for each category
print("SAMPLE POSITIONS BY CATEGORY (for manual verification)\n")
print("="*80)

for category in category_counts.index:
 if category == 'Other':
 continue

 samples = df[df['position_group'] == category]['position'].value_counts().head(10)
 print(f"\n {category} ({category_counts[category]:,} positions)")
 print("-" * 80)
 for i, (pos, count) in enumerate(samples.items(), 1):
 print(f" {i:2d}. {pos:<50} (n={count})")

print("\n" + "="*80)

## 6. Load SBERT Model for Similarity Matching



In [ ]:
# Load SBERT model
print("Loading Sentence-BERT model...")
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
print(" SBERT model loaded!")

## 7. SBERT Similarity Mapping for "Other" Category



In [ ]:
# Define representative examples for each category (for SBERT comparison)
CATEGORY_EXAMPLES = {
 'Executive/Leadership': [
 'Chief Executive Officer',
 'Chief Technology Officer',
 'Chief Financial Officer',
 'Vice President of Engineering',
 'Co-founder and CEO',
 'Chief Marketing Officer',
 ],
 'Management': [
 'Engineering Manager',
 'Product Manager',
 'Team Lead',
 'Director of Operations',
 'Scrum Master',
 'Game Producer',
 ],
 'Mobile Development': [
 'iOS Developer',
 'Android Engineer',
 'Mobile Application Developer',
 'Flutter Developer',
 'React Native Developer',
 ],
 'DevOps': [
 'DevOps Engineer',
 'Site Reliability Engineer',
 'Cloud Infrastructure Engineer',
 'Kubernetes Administrator',
 'MLOps Engineer',
 ],
 'Data/Analytics': [
 'Data Scientist',
 'Machine Learning Engineer',
 'Business Intelligence Analyst',
 'Data Analyst',
 'Business Analyst',
 ],
 'QA/Testing': [
 'QA Engineer',
 'Software Tester',
 'Test Automation Engineer',
 'Quality Assurance Analyst',
 ],
 'Engineering/IT': [
 'Software Engineer',
 'Software Developer',
 'Backend Developer',
 'Frontend Engineer',
 'Full Stack Developer',
 'Database Administrator',
 ],
 'Design/Creative': [
 'UX Designer',
 'UI/UX Designer',
 'Graphic Designer',
 'Product Designer',
 '2D Artist',
 '3D Artist',
 'Animator',
 'Copywriter',
 ],
 'Sales/Marketing': [
 'Sales Manager',
 'Marketing Manager',
 'Business Development Manager',
 'Digital Marketing Specialist',
 'PPC Specialist',
 'Media Buyer',
 'SMM Specialist',
 ],
 'HR/Recruitment': [
 'HR Manager',
 'Recruiter',
 'Talent Acquisition Specialist',
 'Human Resources Business Partner',
 'Agile Coach',
 ],
 'Operations/Admin': [
 'Operations Manager',
 'Administrative Assistant',
 'Office Coordinator',
 'Customer Support Specialist',
 'Customer Care Specialist',
 ],
 'Finance/Legal': [
 'Financial Analyst',
 'Accountant',
 'Legal Counsel',
 'Compliance Officer',
 'Auditor',
 ],
 'Academic/Research': [
 'Research Scientist',
 'Professor',
 'Lecturer',
 'PhD Researcher',
 'Technical Writer',
 ],
 'Healthcare': [
 'Medical Doctor',
 'Nurse',
 'Pharmacist',
 'Healthcare Administrator',
 ],
}

print(f" Defined examples for {len(CATEGORY_EXAMPLES)} categories")

In [ ]:
# Encode category examples
print("Encoding category examples with SBERT...")

category_embeddings = {}
for category, keywords in CATEGORY_KEYWORDS:
 # Use the examples from CATEGORY_EXAMPLES dictionary
 if category in CATEGORY_EXAMPLES:
 examples = CATEGORY_EXAMPLES[category] # Get examples from CATEGORY_EXAMPLES
 embeddings = sbert_model.encode(examples)
 # Use mean embedding as category representation
 category_embeddings[category] = np.mean(embeddings, axis=0)
 else:
 print(f" Warning: No examples defined for {category}")

print(f" Encoded {len(category_embeddings)} category embeddings")

In [ ]:
def sbert_similarity_mapping(position_original, threshold=0.5):
 """
 Map position to category using SBERT cosine similarity.
 Returns category with highest similarity if above threshold, else 'Other'.

 Args:
 position_original: Original position text (not cleaned/stemmed)
 threshold: Minimum similarity score (0-1)
 """
 if pd.isnull(position_original) or str(position_original).strip() == '':
 return 'Other'

 # Encode the position
 pos_embedding = sbert_model.encode([str(position_original)])[0]

 # Calculate similarity to each category
 similarities = {}
 for category, cat_embedding in category_embeddings.items():
 similarity = cosine_similarity(
 pos_embedding.reshape(1, -1),
 cat_embedding.reshape(1, -1)
 )[0][0]
 similarities[category] = similarity

 # Get category with highest similarity
 best_category = max(similarities, key=similarities.get)
 best_similarity = similarities[best_category]

 # Return category if above threshold, else 'Other'
 if best_similarity >= threshold:
 return best_category
 else:
 return 'Other'


print(" SBERT similarity mapping function defined!")

In [ ]:
# Apply SBERT mapping to "Other" positions
other_mask = df['position_group'] == 'Other'
other_count_before = other_mask.sum()

if other_count_before > 0:
 print(f"Applying SBERT similarity mapping to {other_count_before:,} 'Other' positions...")
 print("This may take a few minutes...\n")

 tqdm.pandas(desc="SBERT Mapping")
 df.loc[other_mask, 'position_group_sbert'] = df.loc[other_mask, 'position'].progress_apply(
 sbert_similarity_mapping
 )

 # Update position_group with SBERT results
 df.loc[other_mask, 'position_group'] = df.loc[other_mask, 'position_group_sbert']

 # Statistics
 other_count_after = (df['position_group'] == 'Other').sum()
 mapped_by_sbert = other_count_before - other_count_after

 print(f"\n{'='*60}")
 print("SBERT MAPPING RESULTS")
 print(f"{'='*60}")
 print(f"'Other' before SBERT: {other_count_before:,}")
 print(f"Mapped by SBERT: {mapped_by_sbert:,} ({mapped_by_sbert/other_count_before*100:.2f}%)")
 print(f"'Other' after SBERT: {other_count_after:,} ({other_count_after/len(df)*100:.2f}%)")
 print(f"{'='*60}")
else:
 print(" No 'Other' positions to map with SBERT!")

In [ ]:
# Show examples of positions mapped by SBERT
if mapped_by_sbert > 0:
 print("\n" + "="*80)
 print("EXAMPLES OF POSITIONS MAPPED BY SBERT (not by rules)")
 print("="*80)

 # Get positions that were 'Other' but now have a category
 sbert_mapped_mask = (df['position_group_sbert'] != 'Other') & other_mask
 sbert_mapped_df = df[sbert_mapped_mask].copy()

 # Group by new category and show samples
 for category in sbert_mapped_df['position_group'].value_counts().index:
 count = (sbert_mapped_df['position_group'] == category).sum()
 samples = sbert_mapped_df[sbert_mapped_df['position_group'] == category]['position'].value_counts().head(10)

 print(f"\n {category} ({count:,} positions mapped by SBERT)")
 print("-" * 80)
 for i, (pos, pos_count) in enumerate(samples.items(), 1):
 print(f" {i:2d}. {pos:<60} (n={pos_count})")

 print("\n" + "="*80)

In [ ]:
# Updated category distribution
print("\nFINAL CATEGORY DISTRIBUTION (after Rules + SBERT):\n")
final_counts = df['position_group'].value_counts()
print(final_counts)
print(f"\nTotal categories: {len(final_counts)}")

## 8. DBSCAN Clustering on Remaining "Other" Positions



In [ ]:
# Get remaining "Other" positions
other_positions_df = df[df['position_group'] == 'Other'].copy()

if len(other_positions_df) > 0:
 print(f"Analyzing {len(other_positions_df):,} remaining 'Other' positions with DBSCAN clustering...\n")

 # Get unique positions and clean them
 unique_other_positions = other_positions_df['position'].unique()

 # Filter out any null, empty, or very short positions
 unique_other_positions = [
 str(pos) for pos in unique_other_positions
 if pd.notna(pos) and len(str(pos).strip()) > 2
 ]

 print(f"Unique 'Other' positions: {len(unique_other_positions):,}")

 # Encode positions with error handling
 print("Encoding positions with SBERT...")
 try:
 other_embeddings = sbert_model.encode(
 unique_other_positions,
 show_progress_bar=True,
 convert_to_numpy=True # Force numpy output
 )
 except Exception as e:
 print(f"Error encoding positions: {e}")
 print("\nTrying one-by-one encoding to find problematic position...")

 # Encode one by one to find the problematic one
 other_embeddings = []
 problematic_positions = []

 for pos in tqdm(unique_other_positions, desc="Encoding"):
 try:
 emb = sbert_model.encode([pos], convert_to_numpy=True)
 other_embeddings.append(emb[0])
 except Exception as pos_error:
 print(f" Skipping problematic position: {pos}")
 problematic_positions.append(pos)

 other_embeddings = np.array(other_embeddings)

 # Remove problematic positions from the list
 unique_other_positions = [
 pos for pos in unique_other_positions
 if pos not in problematic_positions
 ]

 print(f"\n Encoded {len(other_embeddings)} positions")
 print(f" Skipped {len(problematic_positions)} problematic positions")

 if len(other_embeddings) > 0:
 # Apply DBSCAN clustering
 print("\nApplying DBSCAN clustering...")
 dbscan = DBSCAN(eps=0.3, min_samples=3, metric='cosine')
 clusters = dbscan.fit_predict(other_embeddings)

 # Create mapping from position to cluster
 position_to_cluster = dict(zip(unique_other_positions, clusters))

 # Cluster statistics
 n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
 n_noise = list(clusters).count(-1)

 print(f"\n{'='*60}")
 print("DBSCAN CLUSTERING RESULTS")
 print(f"{'='*60}")
 print(f"Clusters found: {n_clusters}")
 print(f"Noise points: {n_noise} ({n_noise/len(unique_other_positions)*100:.2f}%)")
 print(f"{'='*60}")

 # Show sample positions from each cluster
 print("\nSample positions from each cluster:\n")
 for cluster_id in sorted(set(clusters)):
 if cluster_id == -1:
 continue # Skip noise

 cluster_positions = [pos for pos, cid in position_to_cluster.items() if cid == cluster_id]
 print(f"Cluster {cluster_id} ({len(cluster_positions)} positions):")
 for pos in cluster_positions[:5]:
 print(f" - {pos}")
 print()

 # Save cluster analysis
 cluster_df = pd.DataFrame({
 'position': unique_other_positions,
 'cluster': clusters
 })
 cluster_df.to_csv('other_positions_clusters.csv', index=False)
 print(" Cluster analysis saved to 'other_positions_clusters.csv'")

else:
 print(" No 'Other' positions remaining for clustering!")

## 9. Final Statistics & Analysis

In [ ]:
# Complete statistics
print("\n" + "="*80)
print(" FINAL GROUPING STATISTICS")
print("="*80)

print(f"\nOriginal unique positions: {df['position'].nunique():,}")
print(f"Final position groups: {df['position_group'].nunique()}")

print(f"\nDistribution by category:")
print(df['position_group'].value_counts())

# Coverage statistics
total = len(df)
other_final = (df['position_group'] == 'Other').sum()
mapped_final = total - other_final

print(f"\nCoverage:")
print(f" Mapped to categories: {mapped_final:,} ({mapped_final/total*100:.2f}%)")
print(f" Remaining 'Other': {other_final:,} ({other_final/total*100:.2f}%)")

print("\n" + "="*80)

In [ ]:
# Sample positions for each final category
print("\nSAMPLE POSITIONS BY FINAL CATEGORY\n")
print("="*80)

for category in df['position_group'].value_counts().index:
 count = (df['position_group'] == category).sum()
 samples = df[df['position_group'] == category]['position'].value_counts().head(5)

 print(f"\n {category} ({count:,} samples)")
 print("-" * 80)
 for i, (pos, pos_count) in enumerate(samples.items(), 1):
 print(f" {i}. {pos:<50} (n={pos_count})")

## 10. Save Results

In [ ]:
# Save grouped dataset
df_final = df[['position', 'position_cleaned', 'position_group', 'Long Description']].copy()
df_final.to_csv('dataset_grouped_positions.csv', index=False)
print(" Grouped dataset saved to 'dataset_grouped_positions.csv'")

# Save position to group mapping
position_mapping = df[['position', 'position_group']].drop_duplicates()
position_mapping = position_mapping.sort_values(['position_group', 'position'])
position_mapping.to_csv('position_to_group_mapping.csv', index=False)
print(f" Position mapping saved to 'position_to_group_mapping.csv' ({len(position_mapping):,} unique positions)")

# Save statistics
stats = {
 'original_unique_positions': df['position'].nunique(),
 'final_groups': df['position_group'].nunique(),
 'total_samples': len(df),
 'mapped_samples': mapped_final,
 'other_samples': other_final,
 'coverage_percentage': mapped_final/total*100,
}

stats_df = pd.DataFrame([stats])
stats_df.to_csv('grouping_statistics.csv', index=False)
print(" Statistics saved to 'grouping_statistics.csv'")

# Save category distribution
category_dist = df['position_group'].value_counts().reset_index()
category_dist.columns = ['category', 'count']
category_dist['percentage'] = (category_dist['count'] / len(df) * 100).round(2)
category_dist.to_csv('category_distribution.csv', index=False)
print(" Category distribution saved to 'category_distribution.csv'")